In [1]:
import pandas as pd
import numpy as np
import gradio as gr
import tempfile
import os
from datetime import datetime

In [2]:
def process_files(a1_file, d1_file):
    try:
        # =========================
        # Load files
        # =========================
        a1_df = pd.read_csv(a1_file.name)
        d1_df = pd.read_csv(d1_file.name)

        # =========================
        # A1 processing
        # =========================
        a1_df['SubmissionDate'] = pd.to_datetime(a1_df['SubmissionDate'], errors='coerce')
        a1_df.sort_values(by='SubmissionDate', ascending=False, inplace=True)

        A_section = ['SubmissionDate', 'A1', 'A1manual'] + [f'A4/{i}' for i in range(1,10)]
        a1_df = a1_df[A_section]

        # Fill ID
        a1_df['A1'] = a1_df['A1'].fillna(a1_df['A1manual'])

        # Convert A4 columns to numeric
        A4_cols = [f'A4/{i}' for i in range(1,10)]
        a1_df[A4_cols] = a1_df[A4_cols].apply(pd.to_numeric, errors='coerce')

        # =========================
        # D1 processing
        # =========================
        d1_df['A1'] = d1_df['A1'].fillna(d1_df['A1manual'])

        useful_column = ['SubmissionDate','A1', 'A1manual', 'SubmitterID', 'SubmitterName', 'Edits']
        d1_df = d1_df[useful_column]

        # =========================
        # Main logic
        # =========================
        result_dict = {}

        for id in d1_df['A1'].dropna().unique():
            tmp_df = a1_df[a1_df['A1'] == id]

            if tmp_df.empty:
                continue

            condition_dict = {}

            for col in A4_cols:
                condition_dict[col] = int(tmp_df[col].sum(skipna=True))

            condition_dict['Record in A1'] = tmp_df.shape[0]

            # Check completeness
            if not all(condition_dict[col] == 1 for col in A4_cols):
                result_dict[id] = condition_dict

        # =========================
        # Convert to DataFrame
        # =========================
        df = pd.DataFrame.from_dict(result_dict, orient='index').reset_index()
        df = df.rename(columns={'index':'A1'})

        if df.empty:
            return "No issues found ✅", None

        # =========================
        # Add remarks
        # =========================
        df['A4_max'] = df[A4_cols].max(axis=1)

        df['Remark'] = np.select(
            [
                df['A4_max'] == 0,
                df['A4_max'] == 2,
                df['A4_max'] == 3,
                df['A4_max'] > 3
            ],
            [
                "All Missing",
                "Double Entry",
                "Triple Entry",
                "More Than 3 Entry"
            ],
            default=""
        )

        df.replace({0: 'Missing'}, inplace=True)
        df.drop(columns=['A4_max'], inplace=True)

        # Split
        df_sr = df[df['A1'].str.contains('SR', na=False)]
        df_pp = df[df['A1'].str.contains('PP', na=False)]

        # =========================
        # Save Excel
        # =========================
        # output_file = tempfile.NamedTemporaryFile(delete=False, suffix=".xlsx")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        # filename = f"A1D1_Check_{timestamp}.xlsx"
        filename = f"A1D1_DataQuality_{datetime.now().strftime('%d%m%Y')}.xlsx"
        output_path = os.path.join(tempfile.gettempdir(), filename)

        # with pd.ExcelWriter(output_file.name, engine='xlsxwriter') as writer:
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            df.to_excel(writer, sheet_name='A1D1_Missing_Info_ALL', index=False)
            df_pp.to_excel(writer, sheet_name='A1D1_Missing_Info_PP', index=False)
            df_sr.to_excel(writer, sheet_name='A1D1_Missing_Info_SR', index=False)

        # return "Processing complete ✅", output_file.name, df
        return "Processing complete ✅", output_path, df, df_pp, df_sr

    except Exception as e:
        return f"Error: {str(e)}", None, None
    


In [3]:
# =========================
# Gradio UI
# =========================
with gr.Blocks(title="A1-D1 Data Validation Tool") as app:

    gr.Markdown("# A1-D1 Data Validation Tool")
    gr.Markdown("Upload A1 and D1 CSV files to check missing and duplicate entries.")

    # =========================
    # Upload section
    # =========================
    with gr.Row():
        a1_input = gr.File(label="Upload A1 Questionnaire CSV")
        d1_input = gr.File(label="Upload D1 Screening CSV")

    # =========================
    # Button
    # =========================
    run_btn = gr.Button("Check")

    # =========================
    # Output section
    # =========================
    with gr.Column():

        with gr.Row():
            status_output = gr.Textbox(label="Status")
            file_output = gr.File(label="Download Result Excel")

        with gr.Column():
            table_output = gr.Dataframe(label="A1D1_Missing_Info_ALL")
            table_output1 = gr.Dataframe(label="A1D1_Missing_Info_PP")
            table_output2 = gr.Dataframe(label="A1D1_Missing_Info_SR")

    # =========================
    # Connect function
    # =========================
    run_btn.click(
        fn=process_files,   # your function
        inputs=[a1_input, d1_input],
        outputs=[status_output, file_output, table_output, table_output1, table_output2]
    )

app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [4]:
# # =========================
# # Gradio UI
# # =========================
# interface = gr.Interface(
#     fn=process_files,
#     inputs=[
#         gr.File(label="Upload A1 Questionnaire CSV"),
#         gr.File(label="Upload D1 Screening CSV")
#     ],
#     outputs=[
#         gr.Textbox(label="Status"),
#         gr.File(label="Download Result Excel"),
#         gr.Dataframe(label="Preview Result")
#     ],
#     title="A1-D1 Data Validation Tool",
#     description="Upload A1 and D1 CSV files to check missing and duplicate entries."
# )

# interface.launch()